# Edit a SLURM header without changing the job body

**Author:** Haoyuan Li

This example parses a Stampede3 batch file, changes selected `#SBATCH` values, and writes a new file. The executable part of the script is preserved exactly.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from gdmate.slurm import SlurmScript

Locate and parse the example relative to this notebook's directory. This works when the notebook is launched from either `notebooks` or the repository root.

In [ ]:
working_directory = Path.cwd()
notebook_directory = (
    working_directory
    if working_directory.name == "notebooks"
    else working_directory / "notebooks"
)
repository_root = notebook_directory.parent
example = repository_root / "scripts/server/stampede3/job_stampede3_2d.slurm"
script = SlurmScript.from_file(example)
[(directive.option, directive.value) for directive in script.directives]

Change only existing header directives. Their original syntax, spacing, and inline comments are retained.

In [ ]:
original_body = script.body
script.set_directive("--partition", "spr")
script.set_directive("-N", "2")
script.set_directive("--job-name", "my_2d_job")
script.header

Write a separate file and verify that reparsing it yields the original body verbatim. A temporary directory keeps the demonstration from leaving generated files in the repository.

In [ ]:
with TemporaryDirectory() as directory:
    output = Path(directory) / "job_stampede3_2d_modified.slurm"
    script.write(output)
    reparsed = SlurmScript.from_file(output)
    assert reparsed.body == original_body
    print(reparsed.text)